# 01 -- Data Audit and Cleaning

**Purpose:** Inspect the raw UDISE+ extract, confirm column availability,
district names, year range, and document what is present vs missing.

No source files are modified. All outputs land in `data/processed/` or `docs/`.

**Key outputs produced here:** `docs/data_log.md`, `docs/data_log.xlsx`

To regenerate the data log:
```
python src/features/build_data_log.py
```

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT    = Path.cwd().parent
V1_PATH = ROOT / 'data/processed/district_year/maharashtra_district_year_shala_score_v1.csv'
df = pd.read_csv(V1_PATH, low_memory=False)
print(f'Shape: {df.shape}')
print(f'Districts: {df["district"].nunique()}')
print(f'Years: {sorted(df["academic_year"].unique())}')

In [ ]:
# Year scope -- full dataset vs spec window vs edge years
years_all  = sorted(df['academic_year'].unique())
years_spec = [y for y in years_all if y not in {'2018-19', '2025-26'}]

scope = pd.DataFrame({
    'Academic_Year':   years_all,
    'In_Spec_Window':  [y in years_spec for y in years_all],
    'Is_Edge_Year':    [y in {'2018-19', '2025-26'} for y in years_all],
    'Districts':       [df[df['academic_year'] == y]['district'].nunique() for y in years_all],
    'Rows':            [len(df[df['academic_year'] == y]) for y in years_all],
})
print('Year scope (full 2018-19 to 2025-26 | spec window 2019-20 to 2024-25 | edge years documented):')
print(scope.to_string(index=False))
print(f'\nTotal rows: {len(df)}  |  Spec window rows: {len(df[df["academic_year"].isin(years_spec)])}')

In [ ]:
# Column availability audit
null_pct = df.isnull().mean().sort_values(ascending=False)
print('Columns with >50% nulls:')
print(null_pct[null_pct > 0.5].to_string())

In [ ]:
# District name check -- current official names (36 total)
print(f'District count: {df["district"].nunique()}')
print(sorted(df['district'].unique()))

In [ ]:
# Confirmed ABSENT columns -- never fabricate these
# These are MISSING_IN_SOURCE: not collected in this UDISE+ release.
MUST_ABSENT = [
    'electricity_pct', 'drinking_water_pct', 'library_pct', 'ramp_pct',
    'internet_pct', 'learning_outcome', 'ger', 'dropout_rate', 'aser_score', 'nas_score',
]
for col in MUST_ABSENT:
    status = 'PRESENT' if col in df.columns else 'ABSENT (MISSING_IN_SOURCE -- confirmed)'
    print(f'  {col:40s} {status}')

In [ ]:
# Confirmed PRESENT columns (0 null) -- the foundation of Shala Score v2
MUST_PRESENT = [
    'ptr_students_per_teacher_raw',
    'facility_functional_toilet_pct',
    'facility_classrooms_good_condition_pct',
    'facility_avg_digital_devices_per_reporting_school',
    'enrollment_reported_enrollment_b_component_total',
    'enrollment_reported_enrollment_g_component_total',
    'enrollment_reported_student_total',
]
print('Key pillar columns -- confirmed present and null count:')
for col in MUST_PRESENT:
    if col in df.columns:
        nulls = df[col].isna().sum()
        print(f'  {col:55s} nulls={nulls}')
    else:
        print(f'  {col:55s} MISSING -- check source file')

In [ ]:
# Data log -- show docs/data_log.md if already generated
data_log_path = ROOT / 'docs' / 'data_log.md'
if data_log_path.exists():
    text = data_log_path.read_text(encoding='utf-8')
    # Print first 60 lines (year scope + district coverage sections)
    for line in text.splitlines()[:60]:
        print(line)
else:
    print('docs/data_log.md not found. Run: python src/features/build_data_log.py')